In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.0
pandas: 3.0.3
저장 폴더: C:\Users\tjwls\OneDrive\문서\ai-data-bootcamp\d009_outputs


In [2]:
# ─────────────────────────────────────────────
# 종합 프로젝트용 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '오염'을 일부러 심어 둡니다: 결측·이상치·표기 혼재·날짜 포맷 혼재·중복·타입 불일치)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 500
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(36, 10, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
        np.random.randint(0, 700, n_customers), unit="D"
    ),
})
# 오염 심기: 나이 이상치, 결측, 지역/멤버십 표기 혼재
customers.loc[[5, 6], "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[[20, 21, 22, 23], "gender"] = np.nan
customers.loc[30, "region"] = " 서울 "
customers.loc[31, "region"] = "Seoul"
customers.loc[40, "membership"] = "VIP"
customers.loc[41, "membership"] = " premium"

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 60
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 199900], n_products),
})

# 3) 주문(orders)
n_orders = 5000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05]),
})

# 날짜 포맷 혼재(문자열로 저장 — 일부러 통일하지 않음)
base_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_orders), unit="D")
date_strings = []
for i, d in enumerate(base_dates):
    if i % 3 == 0:
        date_strings.append(d.strftime("%Y-%m-%d"))
    elif i % 3 == 1:
        date_strings.append(d.strftime("%Y/%m/%d"))
    else:
        date_strings.append(d.strftime("%Y%m%d"))
orders["order_date"] = date_strings

# 오염 심기: 금액 결측, 수량 이상치, 중복 행, 음수 금액(환불로 보이는 값)
orders.loc[np.random.choice(n_orders, 80, replace=False), "amount"] = np.nan
orders.loc[7, "quantity"] = 100
orders.loc[8, "amount"] = -50000.0   # 환불 또는 입력 실수 (판단 필요)
orders = pd.concat([orders, orders.iloc[[0, 1, 2]]], ignore_index=True)  # 중복 3건

# 4) 웹 로그(web_logs) — 텍스트 한 줄로 들어옴(D+005 식 파싱 대상)
n_logs = 1200
log_dates = pd.to_datetime("2025-04-01") + pd.to_timedelta(np.random.randint(0, 60, n_logs), unit="D")
ips = [f"{np.random.randint(1,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}" for _ in range(n_logs)]
paths = np.random.choice(["/home", "/product/123", "/cart", "/checkout", "/search?q=shoes"], n_logs)
web_logs = pd.DataFrame({
    "raw_log": [f"{d.strftime('%Y-%m-%d %H:%M:%S')} - {ip} - GET {p}" for d, ip, p in zip(log_dates, ips, paths)]
})

# CSV로 임시 저장(인제스트 단계에서 읽어옴) — 분석가가 "파일을 받았다" 상황 재현
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)
customers.to_csv(RAW_DIR / "customers.csv", index=False)
products.to_csv(RAW_DIR / "products.csv", index=False)
orders.to_csv(RAW_DIR / "orders.csv", index=False)
web_logs.to_csv(RAW_DIR / "web_logs.csv", index=False)

print("종합 프로젝트 데이터 생성 및 CSV 저장 완료")
print(f"  customers: {customers.shape}")
print(f"  products : {products.shape}")
print(f"  orders   : {orders.shape}  (중복·결측·이상치 포함)")
print(f"  web_logs : {web_logs.shape}")
print(f"\n저장 경로: {RAW_DIR.resolve()}")

종합 프로젝트 데이터 생성 및 CSV 저장 완료
  customers: (500, 6)
  products : (60, 3)
  orders   : (5003, 7)  (중복·결측·이상치 포함)
  web_logs : (1200, 1)

저장 경로: C:\Users\tjwls\OneDrive\문서\ai-data-bootcamp\d009_outputs\raw


In [3]:
# 예제: 방금 저장한 CSV를 다시 읽어 옵니다. (분석가가 새 데이터를 '받은' 상황 재현)
orders_csv = pd.read_csv(RAW_DIR / "orders.csv")
customers_csv = pd.read_csv(RAW_DIR / "customers.csv")
products_csv = pd.read_csv(RAW_DIR / "products.csv")

print("orders_csv.shape :", orders_csv.shape)
print("customers_csv.shape :", customers_csv.shape)
print("products_csv.shape :", products_csv.shape)
print()
print("orders_csv.dtypes")
print(orders_csv.dtypes)

orders_csv.shape : (5003, 7)
customers_csv.shape : (500, 6)
products_csv.shape : (60, 3)

orders_csv.dtypes
order_id           str
customer_id        str
product_id         str
quantity         int64
amount         float64
channel            str
order_date         str
dtype: object


In [4]:
# 예제: customers의 signup_date가 CSV 왕복 후 어떻게 변했는지 확인
print("CSV에서 읽은 직후 dtypes:")
print(customers_csv[["customer_id", "age", "signup_date"]].dtypes)
print()
print("signup_date 첫 5개 값(문자열로 보임):")
print(customers_csv["signup_date"].head().to_list())
print()
# 같은 컬럼을 datetime으로 다시 파싱해야 사용 가능 — 매번 이 작업을 반복해야 합니다.
parsed = pd.to_datetime(customers_csv["signup_date"])
print("파싱 후 dtype:", parsed.dtype)

CSV에서 읽은 직후 dtypes:
customer_id      str
age            int64
signup_date      str
dtype: object

signup_date 첫 5개 값(문자열로 보임):
['2024-07-18', '2025-05-19', '2024-03-27', '2024-02-12', '2025-01-19']

파싱 후 dtype: datetime64[us]


In [7]:
# 스스로 해보자! (1)
# 1) amount, quantity의 dtype을 출력해보세요.
# 2) 결측치(NaN)가 있는 컬럼은 어떤 자료형으로 추론되는지 관찰해보세요.

# 여기에 코드를 작성하세요
print("amount dtype:", orders_csv["amount"].dtype)
print("quantity dtype:", orders_csv["quantity"].dtype)

# amount는 결측치가 섞여 있어 float64로, quantity는 결측치가 없으므로 int64로 추론.

amount dtype: float64
quantity dtype: int64


In [8]:
# 예제: 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report


# orders에 적용
qr_orders = quality_report(orders_csv, "orders_csv")
qr_orders

[품질 리포트] orders_csv
  행 수: 5,003  /  열 수: 7
  메모리: 407.5 KB
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,sample
order_id,str,0,0.0,5000,O00001
customer_id,str,0,0.0,500,C0304
product_id,str,0,0.0,60,P053
quantity,int64,0,0.0,4,3
amount,float64,80,1.6,22,89700.0
channel,str,0,0.0,4,app
order_date,str,0,0.0,540,2025-03-08


In [9]:
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


qr_orders_full = quality_report_full(orders_csv, "orders_csv")
qr_orders_full

[품질 리포트(완전판)] orders_csv
  행 수: 5,003  /  열 수: 7
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
order_id,str,0,0.0,5000,NaN,False
customer_id,str,0,0.0,500,NaN,False
product_id,str,0,0.0,60,NaN,False
quantity,int64,0,0.0,4,0.02,False
amount,float64,80,1.6,22,2.11,False
channel,str,0,0.0,4,NaN,False
order_date,str,0,0.0,540,NaN,False


In [10]:
# 예제: customers와 products에도 같은 함수를 적용 — 한 함수가 어떤 데이터든 진단합니다
print("=" * 60)
qr_customers = quality_report_full(customers_csv, "customers_csv")
display(qr_customers)
print("=" * 60)
qr_products = quality_report_full(products_csv, "products_csv")
display(qr_products)

[품질 리포트(완전판)] customers_csv
  행 수: 500  /  열 수: 6
  완전 중복 행: 0건
  📌 날짜로 보이는 object 컬럼: ['signup_date']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
customer_id,str,0,0.0,500,NaN,False
age,int64,0,0.0,54,1.4,False
gender,str,4,0.8,2,NaN,False
region,str,0,0.0,7,NaN,False
membership,str,0,0.0,5,NaN,False
signup_date,str,0,0.0,355,NaN,True


[품질 리포트(완전판)] products_csv
  행 수: 60  /  열 수: 3
  완전 중복 행: 0건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
product_id,str,0,0.0,60,NaN,False
category,str,0,0.0,5,NaN,False
price,int64,0,0.0,7,0.0,False


In [13]:
# 스스로 해보자! (2)
def quality_report_with_warning(df, name="df", missing_threshold=30.0):
    base = quality_report_full(df, name)
    base["warning"] = np.where(base["missing_pct"] > missing_threshold, "⚠", "")
    return base


# quality_report_with_warning(orders_csv, "orders_csv", missing_threshold=10)

In [ ]:
# 왜 IQR이 D+003에서 봤을 때보다 오늘 더 유용하게 느껴지나요? (힌트: 한 줄로 어떻게 변했나요?) 
# D+006에서 apply로 함수를 컬럼에 적용했습니다. 그 두 도구가 오늘의 진단 함수 안에 함께 들어갔기에.

# 새 데이터셋이 들어와도 이 함수를 그대로 쓸 수 있는 이유는?  
# 함수가 데이터 구조에 독립적이기 때문에, 어떤 데이터셋이든 동일한 방식으로 진단할 수 있습니다.

In [14]:
# 예제 [1단계] 중복 제거 — 가장 먼저
print(f"중복 제거 전: {len(orders_csv):,}행")

# 판단 근거: 완전 동일한 행은 시스템 입력 오류로 간주, 첫 번째만 남깁니다.
orders_step1 = orders_csv.drop_duplicates(keep="first").reset_index(drop=True)

print(f"중복 제거 후: {len(orders_step1):,}행  (제거된 행: {len(orders_csv) - len(orders_step1)}건)")

중복 제거 전: 5,003행
중복 제거 후: 5,000행  (제거된 행: 3건)


In [15]:
# 예제 [2단계] 문자열 정제 — channel, region 표기 통일
print("정제 전 channel 분포:")
print(orders_step1["channel"].value_counts())
print()

# 판단 근거: 앞뒤 공백·대소문자 차이는 같은 값으로 본다 ('APP' → 'app').
orders_step2 = orders_step1.copy()
orders_step2["channel"] = orders_step2["channel"].str.strip().str.lower()

print("정제 후 channel 분포:")
print(orders_step2["channel"].value_counts())

정제 전 channel 분포:
channel
app     2252
web     2251
APP      250
app      247
Name: count, dtype: int64

정제 후 channel 분포:
channel
app    2749
web    2251
Name: count, dtype: int64


In [16]:
# 예제 [3단계] 날짜 파싱 — 3가지 포맷 혼재를 datetime으로 통일
print("정제 전 order_date 자료형:", orders_step2["order_date"].dtype)
print("샘플 값:", orders_step2["order_date"].head(3).to_list())
print()

# 판단 근거: format='mixed'로 자동 추론. 실패하면 NaT(결측 시각)로 표시.
orders_step3 = orders_step2.copy()
orders_step3["order_date"] = pd.to_datetime(
    orders_step3["order_date"], format="mixed", errors="coerce"
)

print("정제 후 order_date 자료형:", orders_step3["order_date"].dtype)
print("NaT(파싱 실패) 건수:", orders_step3["order_date"].isna().sum())
print("샘플 값:", orders_step3["order_date"].head(3).to_list())

정제 전 order_date 자료형: str
샘플 값: ['2025-03-08', '2025/04/21', '20250608']

정제 후 order_date 자료형: datetime64[us]
NaT(파싱 실패) 건수: 0
샘플 값: [Timestamp('2025-03-08 00:00:00'), Timestamp('2025-04-21 00:00:00'), Timestamp('2025-06-08 00:00:00')]


In [17]:
# 예제 [4단계] 이상치 처리 — 비즈니스 규칙 + IQR 혼합
print("이상치 처리 전 amount/quantity 통계:")
print(orders_step3[["amount", "quantity"]].describe().round(0))
print()

# 판단 근거 1: 음수 금액은 환불일 수도 있으나, 이번 분석은 '구매'를 보는 것이므로 별도 보관 후 본 테이블에선 제외.
orders_step4 = orders_step3.copy()
refunds = orders_step4[orders_step4["amount"] < 0].copy()
orders_step4 = orders_step4[(orders_step4["amount"] >= 0) | orders_step4["amount"].isna()]

# 판단 근거 2: quantity=100은 단일 거래로 비현실적. 99 percentile 기준으로 자릅니다.
q99 = orders_step4["quantity"].quantile(0.99)
orders_step4 = orders_step4[orders_step4["quantity"] <= q99]

print(f"음수 금액 분리: {len(refunds)}건 (별도 보관)")
print(f"수량 상위 1% 컷오프: {q99}")
print(f"이상치 처리 후 행 수: {len(orders_step4):,}")
print()
print("이상치 처리 후 quantity 통계:")
print(orders_step4["quantity"].describe().round(2))

이상치 처리 전 amount/quantity 통계:
         amount  quantity
count    4920.0    5000.0
mean   136747.0       2.0
std    128622.0       2.0
min    -50000.0       1.0
25%     29900.0       1.0
50%     99800.0       2.0
75%    199900.0       2.0
max    599700.0     100.0

음수 금액 분리: 1건 (별도 보관)
수량 상위 1% 컷오프: 3.0
이상치 처리 후 행 수: 4,998

이상치 처리 후 quantity 통계:
count    4998.00
mean        1.67
std         0.74
min         1.00
25%         1.00
50%         2.00
75%         2.00
max         3.00
Name: quantity, dtype: float64


In [18]:
# 예제 [5단계] 결측치 처리 — 마지막에 처리
print("결측치 처리 전:")
print(orders_step4.isna().sum())
print()

orders_clean = orders_step4.copy()

# 판단 근거: amount 결측은 '집계 왜곡 방지'를 위해 행 제거. (대안: 컬럼 평균으로 채움 — 분석 목적에 따라 다름)
orders_clean = orders_clean.dropna(subset=["amount"])

print("결측치 처리 후:")
print(orders_clean.isna().sum())
print()
print(f"최종 정제된 orders: {len(orders_clean):,}행")

결측치 처리 전:
order_id        0
customer_id     0
product_id      0
quantity        0
amount         80
channel         0
order_date      0
dtype: int64

결측치 처리 후:
order_id       0
customer_id    0
product_id     0
quantity       0
amount         0
channel        0
order_date     0
dtype: int64

최종 정제된 orders: 4,918행


In [20]:
# 스스로 해보자! (3)
customers_clean = customers_csv.copy()
# 여기에 코드를 작성하세요
customers_clean = customers_csv.drop_duplicates().copy()
print(customers_clean.shape)
customers_clean["region"] = customers_clean["region"].str.strip().replace({"Seoul": "서울"})
print(customers_clean.shape)
customers_clean["membership"] = customers_clean["membership"].str.strip().str.lower()
print(customers_clean.shape)
customers_clean["age"] = customers_clean["age"].where(customers_clean["age"].between(0, 100))
print(customers_clean.shape)
customers_clean = customers_clean.dropna(subset=["age"]).reset_index(drop=True)
print(customers_clean.shape)

print(customers_clean.shape)

(500, 6)
(500, 6)
(500, 6)
(500, 6)
(497, 6)
(497, 6)


In [21]:
# 예제 [1] 테이블 결합 — orders + customers + products
# 판단 근거: orders 기준으로 모두 left join. orders에 매칭 안 되는 고객/상품은 어차피 매출 표에 안 잡힘.

# customers와 products는 Part 3 스타일로 빠르게 정제
customers_clean2 = (
    customers_csv.drop_duplicates()
    .assign(
        region=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
        membership=lambda d: d["membership"].str.strip().str.lower(),
        gender=lambda d: d["gender"].fillna("Unknown"),
    )
    .pipe(lambda d: d[d["age"].between(0, 100)])
    .reset_index(drop=True)
)

merged = (
    orders_clean
    .merge(customers_clean2[["customer_id", "region", "membership", "age"]], on="customer_id", how="left")
    .merge(products[["product_id", "category", "price"]], on="product_id", how="left")
)

print("결합 후 shape:", merged.shape)
print("결합 후 컬럼:", list(merged.columns))
merged.head(3)

결합 후 shape: (4918, 12)
결합 후 컬럼: ['order_id', 'customer_id', 'product_id', 'quantity', 'amount', 'channel', 'order_date', 'region', 'membership', 'age', 'category', 'price']


,order_id,customer_id,product_id,quantity,amount,channel,order_date,region,membership,age,category,price
0,O00001,C0304,P053,3,89700.0,app,2025-03-08,경기,basic,42.0,패션,29900
1,O00002,C0276,P016,1,129900.0,web,2025-04-21,부산,basic,41.0,가전,129900
2,O00003,C0357,P044,2,19800.0,app,2025-06-08,경기,basic,36.0,가전,9900


In [22]:
# 예제 [2] 집계 — 월별 매출 KPI + 카테고리별 객단가
merged_with_month = merged.assign(
    order_month=lambda d: d["order_date"].dt.to_period("M").astype(str)
)

# 월별 KPI
monthly_kpi = (
    merged_with_month
    .groupby("order_month")
    .agg(
        total_revenue=("amount", "sum"),
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        avg_order_value=("amount", "mean"),
    )
    .round(0)
    .reset_index()
)

print("월별 KPI:")
display(monthly_kpi)

월별 KPI:


,order_month,total_revenue,order_count,unique_customers,avg_order_value
0,2025-01,110338200.0,810,393,136220.0
1,2025-02,104596200.0,801,395,130582.0
2,2025-03,114529400.0,843,411,135859.0
3,2025-04,115705000.0,847,416,136606.0
4,2025-05,117985500.0,853,412,138318.0
5,2025-06,109432700.0,764,389,143237.0


In [23]:
# 예제 [3] 파생 컬럼 + 피벗 — 세그먼트별 평균 객단가를 wide 표로
segment_aov = (
    merged_with_month
    .groupby(["membership", "category"])
    .agg(aov=("amount", "mean"))
    .round(0)
    .reset_index()
    .pivot_table(index="membership", columns="category", values="aov")
    .round(0)
)

print("세그먼트(membership) x 카테고리 평균 객단가:")
segment_aov

세그먼트(membership) x 카테고리 평균 객단가:


category,가전,도서,뷰티,식품,패션
membership,,,,,
basic,133922.0,145756.0,115037.0,161290.0,111632.0
premium,120858.0,153347.0,122144.0,166303.0,103757.0
vip,139596.0,183534.0,135818.0,149930.0,106718.0


In [24]:
# 스스로 해보자! (4)
# merged_with_month를 이용해 다음을 구해보세요.
# region(지역)별 월별 매출(total_revenue)을 wide 표로 만들기 (행=region, 열=월)
# 그 표에서 가장 매출이 큰 (지역, 월) 조합 3개를 출력

# 여기에 코드를 작성하세요
region_month = (
    merged_with_month
    .groupby(["region", "order_month"])["amount"].sum()
    .unstack(fill_value=0)
    .round(0)
)
print(region_month)

# 상위 3 조합
top3 = (
    merged_with_month.groupby(["region", "order_month"])["amount"]
    .sum().nlargest(3)
)
print(top3)

order_month     2025-01     2025-02     2025-03     2025-04     2025-05  \
region                                                                    
경기           22963000.0  18254200.0  28366100.0  20731800.0  26278700.0   
대구           17835500.0  25449500.0  21401200.0  24109900.0  18546600.0   
부산           22852800.0  21212400.0  23184300.0  24271200.0  21272900.0   
서울           27429800.0  21792200.0  25819900.0  26668600.0  28305800.0   
인천           18747800.0  17378600.0  15088400.0  19114400.0  20983500.0   

order_month     2025-06  
region                   
경기           24181100.0  
대구           25443400.0  
부산           22354500.0  
서울           21645000.0  
인천           15509200.0  
region  order_month
경기      2025-03        28366100.0
서울      2025-05        28305800.0
        2025-01        27429800.0
Name: amount, dtype: float64


In [25]:
# 예제: 같은 데이터를 CSV / Parquet으로 각각 저장하고 용량·속도·자료형을 비교

# 저장 대상: 정제·변환까지 끝난 merged_with_month
csv_path = OUT_DIR / "merged_with_month.csv"
parquet_path = OUT_DIR / "merged_with_month.parquet"

# 1) 저장 속도 측정
t0 = time.time()
merged_with_month.to_csv(csv_path, index=False)
csv_write = time.time() - t0

t0 = time.time()
merged_with_month.to_parquet(parquet_path, index=False)   # pyarrow 사용
pq_write = time.time() - t0

# 2) 용량 측정
csv_size = csv_path.stat().st_size / 1024
pq_size = parquet_path.stat().st_size / 1024

print(f"[저장 속도]  CSV: {csv_write*1000:.1f}ms  /  Parquet: {pq_write*1000:.1f}ms")
print(f"[파일 용량]  CSV: {csv_size:>7.1f} KB  /  Parquet: {pq_size:>7.1f} KB  (Parquet은 CSV의 {pq_size/csv_size*100:.1f}%)")

[저장 속도]  CSV: 122.4ms  /  Parquet: 314.3ms
[파일 용량]  CSV:   403.0 KB  /  Parquet:    73.4 KB  (Parquet은 CSV의 18.2%)


In [26]:
# 예제: 읽기 속도와 자료형 보존을 비교

t0 = time.time()
from_csv = pd.read_csv(csv_path)
csv_read = time.time() - t0

t0 = time.time()
from_parquet = pd.read_parquet(parquet_path)
pq_read = time.time() - t0

print(f"[읽기 속도]  CSV: {csv_read*1000:.1f}ms  /  Parquet: {pq_read*1000:.1f}ms")
print()
print("CSV에서 다시 읽은 order_date dtype:", from_csv["order_date"].dtype)
print("Parquet에서 다시 읽은 order_date dtype:", from_parquet["order_date"].dtype)

[읽기 속도]  CSV: 103.1ms  /  Parquet: 488.4ms

CSV에서 다시 읽은 order_date dtype: str
Parquet에서 다시 읽은 order_date dtype: datetime64[us]


In [27]:
# 예제: Parquet의 추가 매력 — 컬럼 일부만 빠르게 읽기
# 판단 근거: 다음 분기 분석가는 KPI만 보고 싶을 수 있습니다. 그럴 때 컬럼만 골라 읽으면 빠릅니다.

cols_only = pd.read_parquet(parquet_path, columns=["order_month", "amount", "category"])
print("필요한 3개 컬럼만 읽기:", cols_only.shape)
print(cols_only.head(3))

필요한 3개 컬럼만 읽기: (4918, 3)
  order_month    amount category
0     2025-03   89700.0       패션
1     2025-04  129900.0       가전
2     2025-06   19800.0       가전


In [28]:
# 스스로 해보자! (5)
# 여기에 코드를 작성하세요
csv_p = OUT_DIR / "monthly_kpi.csv"
pq_p = OUT_DIR / "monthly_kpi.parquet"

monthly_kpi.to_csv(csv_p, index=False)
monthly_kpi.to_parquet(pq_p, index=False)

print("CSV     :", csv_p.stat().st_size, "bytes")
print("Parquet :", pq_p.stat().st_size, "bytes")

partial = pd.read_parquet(pq_p, columns=["order_month", "total_revenue"])
print(partial)

CSV     : 300 bytes
Parquet : 3810 bytes
  order_month  total_revenue
0     2025-01    110338200.0
1     2025-02    104596200.0
2     2025-03    114529400.0
3     2025-04    115705000.0
4     2025-05    117985500.0
5     2025-06    109432700.0


In [29]:
# 예제: 각 정제 단계를 작은 함수로 떼어내기 — '한 함수 = 한 책임' 원칙
def drop_duplicates_step(df):
    '''1단계: 완전 중복 행 제거.'''
    return df.drop_duplicates(keep="first").reset_index(drop=True)


def clean_channel_step(df):
    '''2단계: channel의 공백·대소문자 정제.'''
    return df.assign(channel=df["channel"].str.strip().str.lower())


def parse_date_step(df):
    '''3단계: order_date 문자열을 datetime으로 파싱.'''
    return df.assign(
        order_date=pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    )


def remove_outliers_step(df):
    '''4단계: 음수 금액 제거 + 수량 상위 1% 컷오프.'''
    q99 = df["quantity"].quantile(0.99)
    cond = ((df["amount"] >= 0) | df["amount"].isna()) & (df["quantity"] <= q99)
    return df[cond]


def drop_missing_amount_step(df):
    '''5단계: amount 결측 행 제거.'''
    return df.dropna(subset=["amount"]).reset_index(drop=True)


print("5개의 단계 함수가 정의되었습니다. 다음 셀에서 .pipe()로 묶습니다.")

5개의 단계 함수가 정의되었습니다. 다음 셀에서 .pipe()로 묶습니다.


In [ ]:
# 예제: .pipe()로 5단계를 한 흐름으로 묶기 + 시간 측정
t0 = time.time()
orders_clean_v2 = (
    orders_csv
    .pipe(drop_duplicates_step)
    .pipe(clean_channel_step)
    .pipe(parse_date_step)
    .pipe(remove_outliers_step)
    .pipe(drop_missing_amount_step)
)
elapsed = time.time() - t0

print(f"파이프라인 실행: {elapsed*1000:.1f}ms")
print(f"최종 shape: {orders_clean_v2.shape}")
print()
print("이전 셀 단위 결과(orders_clean)와 동일한가?",
      orders_clean_v2.shape == orders_clean.shape)

In [30]:
# 예제: 인제스트~저장까지 전체 파이프라인을 하나의 함수로
def full_pipeline(input_dir: Path, output_path: Path) -> dict:
    '''종합 프로젝트 end-to-end: 인제스트 → 진단 → 정제 → 변환 → 저장.

    돌려주는 dict에는 단계별 행 수, 저장 경로, 보존된 환불 데이터가 들어갑니다.
    '''
    log = {}

    # 1) 인제스트
    orders = pd.read_csv(input_dir / "orders.csv")
    customers = pd.read_csv(input_dir / "customers.csv")
    products = pd.read_csv(input_dir / "products.csv")
    log["ingest_rows"] = {"orders": len(orders), "customers": len(customers), "products": len(products)}

    # 2) 진단
    log["quality_before"] = orders.isna().sum().to_dict()

    # 3) 정제 (위에서 정의한 단계 함수 재사용)
    orders_c = (
        orders
        .pipe(drop_duplicates_step)
        .pipe(clean_channel_step)
        .pipe(parse_date_step)
        .pipe(remove_outliers_step)
        .pipe(drop_missing_amount_step)
    )
    customers_c = (
        customers.drop_duplicates()
        .assign(
            region=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
            membership=lambda d: d["membership"].str.strip().str.lower(),
            gender=lambda d: d["gender"].fillna("Unknown"),
        )
        .pipe(lambda d: d[d["age"].between(0, 100)])
        .reset_index(drop=True)
    )

    # 4) 변환
    merged = (
        orders_c
        .merge(customers_c[["customer_id", "region", "membership", "age"]], on="customer_id", how="left")
        .merge(products[["product_id", "category", "price"]], on="product_id", how="left")
        .assign(order_month=lambda d: d["order_date"].dt.to_period("M").astype(str))
    )

    # 5) 저장
    merged.to_parquet(output_path, index=False)
    log["saved_to"] = str(output_path)
    log["final_rows"] = len(merged)
    return log


# 함수 실행
pipeline_log = full_pipeline(RAW_DIR, OUT_DIR / "pipeline_result.parquet")
for k, v in pipeline_log.items():
    print(f"{k}: {v}")

ingest_rows: {'orders': 5003, 'customers': 500, 'products': 60}
quality_before: {'order_id': 0, 'customer_id': 0, 'product_id': 0, 'quantity': 0, 'amount': 80, 'channel': 0, 'order_date': 0}
saved_to: d009_outputs\pipeline_result.parquet
final_rows: 4918


In [31]:
# 분기 ① Polars 모드 — 같은 정제 흐름을 Lazy로 표현 (D+007 도구 재활용)
# 오늘 데이터는 작아 속도 이득은 미미하지만, '같은 흐름이 다른 도구로 표현됨'을 확인합니다.
try:
    import polars as pl

    def polars_pipeline_demo(csv_path):
        '''Polars Lazy 파이프라인 — pandas .pipe()와 같은 정제를 표현만 다르게.'''
        return (
            pl.scan_csv(csv_path)                                   # lazy: 아직 안 읽음
              .unique()                                              # 중복 제거
              .with_columns(
                  pl.col("channel").str.strip_chars().str.to_lowercase(),
              )
              .filter(
                  (pl.col("amount") >= 0) | pl.col("amount").is_null()
              )
              .drop_nulls(subset=["amount"])
              .collect()                                              # eager: 이때 실제 실행
        )

    orders_polars = polars_pipeline_demo(RAW_DIR / "orders.csv")
    print(f"Polars 결과 shape: {orders_polars.shape}")
    print("(merge·집계 전 단계라 pandas 최종본과 행 수가 다를 수 있어요 —")
    print(" 핵심은 같은 정제 흐름의 *다른 표현*입니다.)")
except ImportError:
    print("polars 미설치 — 'pip install polars'로 설치하면 이 분기를 켤 수 있습니다.")

Polars 결과 shape: (4919, 7)
(merge·집계 전 단계라 pandas 최종본과 행 수가 다를 수 있어요 —
 핵심은 같은 정제 흐름의 *다른 표현*입니다.)


In [32]:
# 분기 ② 수치 스케일링 — 정제 결과를 모델 입력용으로 표준화 (D+006 scaler 재활용)
from sklearn.preprocessing import StandardScaler

merged_loaded = pd.read_parquet(OUT_DIR / "pipeline_result.parquet")
num_cols = [c for c in ["amount", "quantity", "price", "age"] if c in merged_loaded.columns]

scaler = StandardScaler()
scaled_in = merged_loaded[num_cols].fillna(merged_loaded[num_cols].median(numeric_only=True))
scaled_arr = scaler.fit_transform(scaled_in)
scaled_df = pd.DataFrame(scaled_arr, columns=num_cols)

print("스케일링 전 (mean/std):")
print(merged_loaded[num_cols].describe().loc[["mean", "std"]].round(2))
print("\n스케일링 후 (mean≈0, std≈1):")
print(scaled_df.describe().loc[["mean", "std"]].round(2))

스케일링 전 (mean/std):
         amount  quantity     price    age
mean  136760.27      1.67  81601.91  36.07
std   128608.86      0.74  62418.31   9.81

스케일링 후 (mean≈0, std≈1):
      amount  quantity  price  age
mean     0.0      -0.0    0.0  0.0
std      1.0       1.0    1.0  1.0


In [33]:
# 스스로 해보자! (6)
def full_pipeline_v2(input_dir, output_path):
    log = {}
    orders = pd.read_csv(input_dir / "orders.csv")
    customers = pd.read_csv(input_dir / "customers.csv")
    products = pd.read_csv(input_dir / "products.csv")

    orders_c = (orders.pipe(drop_duplicates_step).pipe(clean_channel_step)
                .pipe(parse_date_step).pipe(remove_outliers_step)
                .pipe(drop_missing_amount_step))

    log["row_change"] = {"before": len(orders), "after": len(orders_c)}

    merged = (
        orders_c
        .merge(customers, on="customer_id", how="left")
        .merge(products[["product_id", "category"]], on="product_id", how="left")
        .assign(order_month=lambda d: d["order_date"].dt.to_period("M").astype(str))
    )
    log["monthly_revenue"] = merged.groupby("order_month")["amount"].sum().round(0).to_dict()

    merged.to_parquet(output_path, index=False)
    return log
# full_pipeline_v2(RAW_DIR, OUT_DIR / "pipeline_v2.parquet")

In [34]:
# 예제: 결정 로그를 리스트로 누적하다가 마지막에 표로 출력
decisions = []


def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})


# Part 3에서 했던 결정을 손으로 기록 (실무에서는 단계 함수 안에서 호출)
log_decision(
    "1. 중복 제거",
    "완전 중복 행 drop",
    "동일 키 동일 값은 시스템 입력 오류로 간주",
    "3건 제거",
)
log_decision(
    "2. 문자열 정제",
    "channel을 lower/strip",
    "'app '·'APP'은 'app'과 같은 채널",
    "채널 종류 4→2로 통일",
)
log_decision(
    "3. 날짜 파싱",
    "order_date를 datetime으로",
    "월별 집계가 분석 목적이므로 datetime 필요",
    f"NaT(파싱 실패): {orders_step3['order_date'].isna().sum()}건",
)
log_decision(
    "4. 이상치 처리",
    "음수 amount 분리, 수량 상위 1% 컷오프",
    "매출 분석이 목적이므로 환불은 별도 보관",
    f"환불 {len(refunds)}건, 수량 상위 컷오프 {q99}",
)
log_decision(
    "5. 결측치 처리",
    "amount 결측 행 제거",
    "결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",
    "최종 80건 내외 제거",
)

decision_log = pd.DataFrame(decisions)
decision_log

,step,action,reason,result
0,1. 중복 제거,완전 중복 행 drop,동일 키 동일 값은 시스템 입력 오류로 간주,3건 제거
1,2. 문자열 정제,channel을 lower/strip,'app '·'APP'은 'app'과 같은 채널,채널 종류 4→2로 통일
2,3. 날짜 파싱,order_date를 datetime으로,월별 집계가 분석 목적이므로 datetime 필요,NaT(파싱 실패): 0건
3,4. 이상치 처리,"음수 amount 분리, 수량 상위 1% 컷오프",매출 분석이 목적이므로 환불은 별도 보관,"환불 1건, 수량 상위 컷오프 3.0"
4,5. 결측치 처리,amount 결측 행 제거,"결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지",최종 80건 내외 제거


In [35]:
# 예제: 결정 로그를 마크다운 표로 출력해 노트북 끝(또는 별도 파일)에 박기
md_table = "| 단계 | 액션 | 근거 | 결과 |\n|---|---|---|---|\n"
for d in decisions:
    md_table += f"| {d['step']} | {d['action']} | {d['reason']} | {d['result']} |\n"

print(md_table)

| 단계 | 액션 | 근거 | 결과 |
|---|---|---|---|
| 1. 중복 제거 | 완전 중복 행 drop | 동일 키 동일 값은 시스템 입력 오류로 간주 | 3건 제거 |
| 2. 문자열 정제 | channel을 lower/strip | 'app '·'APP'은 'app'과 같은 채널 | 채널 종류 4→2로 통일 |
| 3. 날짜 파싱 | order_date를 datetime으로 | 월별 집계가 분석 목적이므로 datetime 필요 | NaT(파싱 실패): 0건 |
| 4. 이상치 처리 | 음수 amount 분리, 수량 상위 1% 컷오프 | 매출 분석이 목적이므로 환불은 별도 보관 | 환불 1건, 수량 상위 컷오프 3.0 |
| 5. 결측치 처리 | amount 결측 행 제거 | 결측은 결제 실패 로그로 추정(MCAR 가정), 매출 왜곡 방지 | 최종 80건 내외 제거 |



In [36]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


In [37]:
# 시나리오 1 — 모범 답안
qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
store_id,str,0,0.00,7,NaN,False
date_str,str,0,0.00,9,NaN,False
item,str,0,0.00,6,NaN,False
quantity,int64,0,0.00,5,14.56,False
unit_price,int64,0,0.00,5,0.00,False
revenue,float64,60,2.77,26,17.30,False


In [42]:
# 시나리오 2 — 모범 답안
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True)
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip())
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"}))
def b_parse_date(df): return df.assign(
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"])

refunds_bakery = bakery[bakery["revenue"] < 0].copy()

bakery_clean = (
    bakery
    .pipe(b_dedup)
    .pipe(b_strip_store)
    .pipe(b_unify_item)
    .pipe(b_parse_date)
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외
    .pipe(lambda d: d.dropna(subset=["revenue"])) # 결측 제거
    .reset_index(drop=True)
)

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")
print("결측치 처리 전:")
print(bakery.isna().sum())
print("결측치 처리 후:")
print(bakery_clean.isna().sum())
print()

정제 전: (2164, 6)  →  정제 후: (1784, 6)
환불 보관: 1건
item 종류: <ArrowStringArray>
['쿠키', '케이크', '샌드위치', '크루아상', '식빵']
Length: 5, dtype: str
결측치 처리 전:
store_id       0
date_str       0
item           0
quantity       0
unit_price     0
revenue       60
dtype: int64
결측치 처리 후:
store_id      0
item          0
quantity      0
unit_price    0
revenue       0
date          0
dtype: int64



In [39]:
# 시나리오 3 — 모범 답안

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6834000.0,7243500.0,5884000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S04,6471500.0,7105000.0,6744500.0
S05,8681500.0,4102000.0,4960000.0
S06,9156000.0,5095500.0,6796500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: C:\Users\tjwls\OneDrive\문서\ai-data-bootcamp\d009_outputs


In [48]:
# 시나리오 3 — 모범 답안

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
    .sort_values("2025-04", ascending=False)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S06,9156000.0,5095500.0,6796500.0
S05,8681500.0,4102000.0,4960000.0
S02,7531500.0,6768500.0,7996500.0
S03,7424500.0,4548000.0,7297500.0
S01,6834000.0,7243500.0,5884000.0
S04,6471500.0,7105000.0,6744500.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6878.0,26690000.0,365
샌드위치,6555.0,26297000.0,370
케이크,6859.0,23788500.0,359
식빵,6555.0,22997500.0,355
크루아상,6946.0,20867500.0,335



Parquet 저장 완료: C:\Users\tjwls\OneDrive\문서\ai-data-bootcamp\d009_outputs


In [ ]:
# 코드 퀴즈 — 모범 답안
cake_kpi = (
    bakery_clean
    .query("item == '케이크' and revenue >= 10000")
    .groupby("store_id")["revenue"]
    .mean()
    .round(0)
    .reset_index(name="avg_cake_revenue")
)
print(cake_kpi) 

cake_kpi.to_parquet(OUT_DIR / "cake_kpi.parquet", index=False)
print(f"\n저장 완료: {OUT_DIR / 'cake_kpi.parquet'}")

  store_id  avg_cake_revenue
0      S01           81722.0
1      S02          134550.0
2      S03           99488.0
3      S04           85946.0
4      S05          104395.0
5      S06           71805.0

저장 완료: d009_outputs\cake_kpi.parquet


# 빵셀러 데이터 정제·집계 보고서 

## 1. 데이터 개요
- 출처: (가상) 빵셀러 베이커리 체인 운영 로그
- 기간: 2025-04-01 ~ 2025-06-30 (3개월)
- 원본 행 수 / 정제 후 행 수: (2164)행 → (1784)행

## 2. 발견한 품질 문제
- 결측: revenue 2.77%
- 중복: 321
- 이상치: quantity 14.56% / revenue 17.30%
- 표기 혼재: item '케이크' vs '케익'

## 3. 처리 결정과 근거 (5줄로)
1. 중복 행 (321)건 → 완전 동일한 행은 시스템 입력 오류로 간주, 첫 번째만 남깁니다.
2. store_id 공백 → 제거
3. item '케익' → '케이크' → 케이크로 통일
4. revenue 음수 → 별도 보관 (1)건
5. revenue 결측 (60)건 → 제거 (이유: '집계 왜곡 방지'를 위해 행 제거.)

## 4. 주요 KPI 결과 (2줄)
- (월·매장 관점) 가장 매출이 큰 매장과 그 월: 2025-04 "S06"
- (상품 관점) 매출 1위 상품: 쿠키

## 5. 한계와 후속 작업
- 가장 평균 매출이 큰 매장과 그 월 분석 필요
- 결측 원인 분석 필요
- 다른 연도, 혹은 더 많은 날짜 데이터 수집 필요